## Enrichir les données APLs avec le taux de mortalité

In [1]:
import pandas as pd
from pathlib import Path

# On essaie d'utiliser __file__, sinon on prend le dossier actuel
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

ROOT = BASE_DIR.parent
DATA_DIR = ROOT / "data"

%ls ../data/extracted/
print('-'*30)
%ls ../data/processed/data/commune

communes_31_geom_simplified.parquet
communes_31_geom_simplified_geo.parquet
communes_densite.csv
dept_taux_mortalité_2024.parquet
mortalite_departements.csv
raw_comm_acces_urgences.csv
raw_comm_acces_urgences.parquet
raw_dept_taux_mortalité_2024.csv
raw_dept_taux_mortalité_2024.parquet
------------------------------
commune_clusters.parquet               raw_comm_score_apl_niv1.parquet
commune_niv1_score_irdes.csv           raw_comm_score_apl_niv2.parquet
commune_niv1_score_irdes.parquet       score_sante_territoires_final.csv
raw_comm_score_apl_niv0.csv            score_sante_territoires_final.parquet
raw_comm_score_apl_niv0.parquet


In [2]:
# data/processed/data/commune/raw_comm_score_apl_niv1.csv
file_path = DATA_DIR / 'processed/data/commune/raw_comm_score_apl_niv2.parquet'

df_apl = pd.read_parquet(file_path)

df_apl.head()

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,apl_kines_std,apl_sagesfemmes_std,score_apl,quintile_apl_nat,population,superficie,code_insee_du_departement,codes_siren_des_epci,code_insee_de_la_region,tps_SU_SMUR
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832.0,35.270,122.416,...,-0.610034,1.186782,-0.317946,Q2 (faible),859.0,1590.0,01,200069193,84,31.0
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267.0,36.427,109.849,...,-0.510025,-0.214993,-0.291066,Q2 (faible),273.0,920.0,01,240100883,84,18.0
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854.0,59.621,201.121,...,1.084761,1.284043,0.778222,Q5 (très bon),15554.0,2460.0,01,240100883,84,0.0
3,01005,Ambérieux-en-Dombes,3.706,3.648,3.015,2.998,1852.496,1897.0,50.539,131.876,...,0.774416,1.587885,0.583430,Q5 (très bon),1917.0,1590.0,01,200042497,84,23.5
4,01006,Ambléon,0.889,0.775,0.648,0.570,121.314,113.0,10.929,44.227,...,-1.130735,-0.453044,-1.231006,Q1 (très faible),114.0,590.0,01,200040350,84,15.0


In [3]:
# data/extracted/raw_dept_taux_mortalité_2024.parquet

file_path = DATA_DIR / 'extracted/raw_dept_taux_mortalité_2024.parquet'

df_txmor = pd.read_parquet(file_path)

df_txmor.head()

,Code,Département,taux_brut,taux_femmes,taux_hommes,taux_prematuré,taux_65plus,taux_infantile,nb_deces
0,01,Ain,7.9,7.8,7.9,1.4,36.4,3.6,5283.0
1,02,Aisne,11.0,10.7,11.3,2.2,41.8,4.0,5822.0
2,03,Allier,13.5,13.4,13.7,2.1,37.5,2.8,4545.0
3,04,Alpes-de-Haute-Provence,11.8,11.5,12.2,1.8,36.0,3.2,1982.0
4,05,Hautes-Alpes,10.6,10.2,10.9,1.7,31.1,2.5,1519.0


In [4]:
df_txmor.columns

Index(['Code', 'Département', 'taux_brut', 'taux_femmes', 'taux_hommes',
       'taux_prematuré', 'taux_65plus', 'taux_infantile', 'nb_deces'],
      dtype='object')

In [5]:
df_apl['code_insee_du_departement'] = df_apl['code_insee_du_departement'].astype(str)
df_txmor['Code'] = df_txmor['Code'].astype(str)

In [6]:
cols_to_add = ['Code', 'taux_brut', 'taux_prematuré']

df_final = pd.merge(
    df_apl,
    df_txmor[cols_to_add],
    left_on='code_insee_du_departement',
    right_on='Code',
    how='left'
)


In [7]:
df_final.head()

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,quintile_apl_nat,population,superficie,code_insee_du_departement,codes_siren_des_epci,code_insee_de_la_region,tps_SU_SMUR,Code,taux_brut,taux_prematuré
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832.0,35.270,122.416,...,Q2 (faible),859.0,1590.0,01,200069193,84,31.0,01,7.9,1.4
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267.0,36.427,109.849,...,Q2 (faible),273.0,920.0,01,240100883,84,18.0,01,7.9,1.4
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854.0,59.621,201.121,...,Q5 (très bon),15554.0,2460.0,01,240100883,84,0.0,01,7.9,1.4
3,01005,Ambérieux-en-Dombes,3.706,3.648,3.015,2.998,1852.496,1897.0,50.539,131.876,...,Q5 (très bon),1917.0,1590.0,01,200042497,84,23.5,01,7.9,1.4
4,01006,Ambléon,0.889,0.775,0.648,0.570,121.314,113.0,10.929,44.227,...,Q1 (très faible),114.0,590.0,01,200040350,84,15.0,01,7.9,1.4


In [8]:
df_final = df_final.drop(columns=['Code'])

In [9]:
df_final.rename(columns={'taux_brut':'tx_mortalite',
                         'taux_prematuré':'tx_mort_premature'}, inplace=True)

In [10]:
df_final.columns

Index(['code_insee_comm', 'nom_commune', 'apl_medecins', 'apl_65ans',
       'apl_62ans', 'apl_60ans', 'pop_standard_2021', 'pop_totale_2021',
       'apl_dentistes', 'apl_infirmiers', 'apl_kines', 'apl_sagesfemmes',
       'apl_medecins_2022', 'apl_dentistes_2022', 'apl_infirmiers_2022',
       'apl_kines_2022', 'apl_sagesfemmes_2022', 'delta_apl_medecins',
       'delta_apl_dentistes', 'delta_apl_infirmiers', 'delta_apl_kines',
       'delta_apl_sagesfemmes', 'apl_medecins_std', 'apl_dentistes_std',
       'apl_infirmiers_std', 'apl_kines_std', 'apl_sagesfemmes_std',
       'score_apl', 'quintile_apl_nat', 'population', 'superficie',
       'code_insee_du_departement', 'codes_siren_des_epci',
       'code_insee_de_la_region', 'tps_SU_SMUR', 'tx_mortalite',
       'tx_mort_premature'],
      dtype='object')

In [11]:
file_path = '../data/processed/data/commune/raw_comm_score_apl_niv3.parquet'
df_final.to_parquet(file_path)